# Lab 3: Experimentación de hiperparámetros con CIFAR-10



El objetivo de este laboratorio es experimentar con los conceptos teóricos vistos en clase. Se propone seguir la estructura de experimentos del documento. Como hemos visto durante el tema es muy importante vuestra conclusión después del experimento.

Para evaluar con cual nos quedamos después de cada experimento vamos a quedarnos con el que tenga mejor Accuracy en los datos de validación.

El dataset a utilizar es **CIFAR-10**, un dataset ampliamente utilizado en visión por computador que contiene 60.000 imágenes en color de 32x32 píxeles, divididas en 10 clases: avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco y camión. El dataset viene incluido directamente en Keras, por lo que no es necesario descargarlo manualmente.

Cada clase tiene 6.000 imágenes, con 50.000 imágenes de entrenamiento y 10.000 de test. A pesar de la baja resolución de las imágenes, CIFAR-10 es un dataset desafiante ya que los objetos pueden aparecer en diferentes posiciones, ángulos y con fondos variados.


## Carga de los datos

In [ ]:
import numpy as np
import keras
from keras.datasets import cifar10
import matplotlib.pyplot as plt

# Keras descarga y divide automáticamente el dataset CIFAR-10 en datos de entrenamiento y prueba
print("Descargando/Cargando el dataset CIFAR-10...")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Verificamos las dimensiones de los datos para confirmar que todo está correcto
print("--- Datos cargados con éxito ---")
print(f"Imágenes de entrenamiento: {x_train.shape}")   # (50000, 32, 32, 3)
print(f"Etiquetas de entrenamiento: {y_train.shape}")   # (50000, 1)
print(f"Imágenes de prueba: {x_test.shape}")            # (10000, 32, 32, 3)
print(f"Etiquetas de prueba: {y_test.shape}")            # (10000, 1)


In [ ]:
# Mapeo de las 10 clases de CIFAR-10: cada índice corresponde a una categoría de objeto
MAP_CHARACTERS = {
    0: 'Avión',
    1: 'Automóvil',
    2: 'Pájaro',
    3: 'Gato',
    4: 'Ciervo',
    5: 'Perro',
    6: 'Rana',
    7: 'Caballo',
    8: 'Barco',
    9: 'Camión'
}

# Las imágenes de CIFAR-10 ya tienen un tamaño estándar de 32x32 píxeles
# Usamos este tamaño directamente para la entrada de los modelos
IMG_SIZE = 32
NUM_CLASSES = 10


In [ ]:
# Barajamos aleatoriamente los datos de entrenamiento.
# Aunque CIFAR-10 ya viene bastante mezclado, es una buena práctica
# asegurarnos de que los datos están en orden aleatorio para que la
# partición en entrenamiento/validación sea representativa.
perm = np.random.permutation(len(x_train))
x_train, y_train = x_train[perm], y_train[perm]


## Herramientas de visualización de resultados

In [ ]:
# Definición de funciones que permitirán la visualización de las gráficas de entrenamiento
def plot_acc(history, title="Model Accuracy"):
    """Imprime una gráfica mostrando la accuracy por epoch obtenida en un entrenamiento"""
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento', 'Validación'], loc='upper left')
    plt.show()
    
def plot_loss(history, title="Model Loss"):
    """Imprime una gráfica mostrando la pérdida por epoch obtenida en un entrenamiento"""
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(title)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento', 'Validación'], loc='upper right')
    plt.show()
    
def plot_compare_losses(history1, history2, name1="Red 1",
                        name2="Red 2", title="Graph title"):
    """Compara losses de dos entrenamientos con nombres name1 y name2"""
    plt.figure(figsize=(10, 6))
    plt.plot(history1.history['loss'], color="green")
    plt.plot(history1.history['val_loss'], 'r--', color="green")
    plt.plot(history2.history['loss'], color="blue")
    plt.plot(history2.history['val_loss'], 'r--', color="blue")
    plt.title(title)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento ' + name1, 'Validación ' + name1, 
                'Entrenamiento ' + name2, 'Validación ' + name2],
               loc='upper right')
    plt.show()
    
def plot_compare_accs(history1, history2, name1="Red 1",
                      name2="Red 2", title="Graph title"):
    """Compara accuracies de dos entrenamientos con nombres name1 y name2"""
    plt.figure(figsize=(10, 6))
    plt.plot(history1.history['accuracy'], color="green")
    plt.plot(history1.history['val_accuracy'], 'r--', color="green")
    plt.plot(history2.history['accuracy'], color="blue")
    plt.plot(history2.history['val_accuracy'], 'r--', color="blue")
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train ' + name1, 'Val ' + name1, 
                'Train ' + name2, 'Val ' + name2], 
               loc='lower right')
    plt.show()


## Cosas a tener en cuenta:

A continuación se detallan una serie de aspectos orientativos que podrían ser analizados en vuestro informe

*   Realizar un análisis de los datos a utilizar.
* Recuerda partir los datos en training/validation para tener una buena estimación de los valores que nuestro modelo tendrá en los datos de test, así como comprobar que no estamos cayendo en overfitting. Una posible partición puede ser 80 / 20.
* Las imágenes **no están normalizadas**. Hay que normalizarlas como hemos hecho en trabajos anteriores.
* Un error común en Keras es no instanciar un nuevo modelo cada vez que hacemos un nuevo entrenamiento. Al hacer

      *model = Sequential()*
      *model.add(lo que sea)  # Definición del modelo*
      *model.fit()*

    Si queremos entrenar un nuevo modelo o el mismo modelo otra vez, es necesario volver a inicializar el modelo con model = Sequential().
    Si olvidamos este paso y volvemos a hacer fit(), el modelo seguirá entrenando por donde se quedó en el último fit().
* Se recomienda construir una tabla con el mejor valor del accuracy y función de validación.
* Vamos a utilizar la misma arquitectura de red neuronal para todos los experimentos, que mostramos a continuación.

In [ ]:
from keras import layers
from keras import models
from keras.optimizers import Adamax, RMSprop, SGD
from keras.callbacks import EarlyStopping

# Definición y construcción del modelo base.
# Arquitectura: Conv2D -> MaxPooling -> Flatten -> Dense -> Dense -> Output
# Se usa input_shape=(32, 32, 3) porque las imágenes de CIFAR-10 son de 32x32 con 3 canales (RGB)
model = models.Sequential()
model.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
# Capa de salida con 10 neuronas (una por cada clase de CIFAR-10) y activación softmax
model.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))
model.summary()


## Realización de los experimentos

### Experimento 1: Visualización y preparación del dataset

  * Visualizar algunas imágenes aleatoriamente. 
  * Comprobar el número de imágenes y formato.
  * Normalizar.
  * Cualquier otra acción que consideres oportuna que enriquezca el experimento.

Primeramente vamos a visualizar aleatoriamente algunas imágenes del dataset de training junto con su etiqueta.

In [ ]:
# Visualización de 16 imágenes aleatorias del dataset de entrenamiento
# Mostramos una cuadrícula 4x4 con la etiqueta de cada imagen
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
indices = np.random.choice(len(x_train), 16, replace=False)

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[indices[i]])
    # y_train tiene shape (N, 1), así que usamos [0] para acceder al valor
    label = int(y_train[indices[i]][0])
    ax.set_title(MAP_CHARACTERS[label], fontsize=10)
    ax.axis('off')

plt.suptitle('Muestra aleatoria de imágenes CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Comprobación del número de imágenes y formato de los datos
print("=== Información del dataset ===")
print(f"Número de imágenes de entrenamiento: {x_train.shape[0]}")
print(f"Número de imágenes de test: {x_test.shape[0]}")
print(f"Dimensión de cada imagen: {x_train.shape[1:]}")
print(f"Tipo de dato de las imágenes: {x_train.dtype}")
print(f"Valor mínimo de píxel: {x_train.min()}")
print(f"Valor máximo de píxel: {x_train.max()}")
print(f"Número de clases: {len(MAP_CHARACTERS)}")

# Distribución de clases en el conjunto de entrenamiento
unique, counts = np.unique(y_train, return_counts=True)
print("\n=== Distribución de clases (entrenamiento) ===")
for u, c in zip(unique, counts):
    print(f"  {MAP_CHARACTERS[u]}: {c} imágenes")


In [ ]:
# Normalización de las imágenes: convertimos los valores de los píxeles
# del rango [0, 255] al rango [0, 1] dividiendo por 255.0
# Esto facilita el entrenamiento de las redes neuronales al mantener
# los valores en un rango más controlado.
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Verificamos que la normalización se ha realizado correctamente
print(f"Rango de valores tras normalización: [{x_train.min()}, {x_train.max()}]")
print(f"Tipo de dato tras normalización: {x_train.dtype}")


### Experimento 2: Relu vs Tangente hiperbólica

Para la realización de este experimento tiene que utilizar los siguientes hiperparámetros:

*   Optimizer: SGD
*   Loss: sparse_categorical_crossentropy
*   Metrics: accuracy
*   EarlyStopping
      *   monitor=val_loss
      *   patience = 2
      *   verbose=1
*   Batch_size: 32



Construcción del modelo Relu.

In [ ]:
# ---- MODELO CON ACTIVACIÓN ReLU ----
# Construimos el modelo con la misma arquitectura base usando ReLU como función de activación
model_relu = models.Sequential()
model_relu.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_relu.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_relu.add(layers.Flatten())
model_relu.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_relu.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_relu.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

# Compilamos con SGD y sparse_categorical_crossentropy (las etiquetas son enteros, no one-hot)
model_relu.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# EarlyStopping: detiene el entrenamiento si la val_loss no mejora en 2 epochs consecutivos
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

# Entrenamos con batch_size=32 y validation_split=0.2 (80% train, 20% validación)
history_relu = model_relu.fit(x_train, y_train, epochs=20, batch_size=32,
                               validation_split=0.2, callbacks=[early_stop])


In [ ]:
# ---- MODELO CON ACTIVACIÓN TANH ----
# Construimos el mismo modelo pero usando tangente hiperbólica (tanh) como activación
model_tanh = models.Sequential()
model_tanh.add(layers.Conv2D(64, (2, 2), activation='tanh', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_tanh.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_tanh.add(layers.Flatten())
model_tanh.add(layers.Dense(128, activation='tanh', name='Hidden-Layer-1'))
model_tanh.add(layers.Dense(64, activation='tanh', name='Hidden-Layer-2'))
model_tanh.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

# Compilamos con los mismos hiperparámetros que el modelo ReLU
model_tanh.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_tanh = model_tanh.fit(x_train, y_train, epochs=20, batch_size=32,
                               validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparación visual de los resultados entre ReLU y Tanh
plot_compare_losses(history_relu, history_tanh, name1="ReLU", name2="Tanh",
                    title="Comparación de Loss: ReLU vs Tanh")
plot_compare_accs(history_relu, history_tanh, name1="ReLU", name2="Tanh",
                  title="Comparación de Accuracy: ReLU vs Tanh")

# Mostramos las métricas finales de validación de cada modelo
print(f"ReLU - Val Accuracy: {max(history_relu.history['val_accuracy']):.4f}, Val Loss: {min(history_relu.history['val_loss']):.4f}")
print(f"Tanh - Val Accuracy: {max(history_tanh.history['val_accuracy']):.4f}, Val Loss: {min(history_tanh.history['val_loss']):.4f}")


**Conclusión Experimento 2:** Se espera que ReLU tenga un mejor rendimiento que Tanh. ReLU es más eficiente computacionalmente y menos susceptible al problema del desvanecimiento del gradiente (vanishing gradient), lo cual permite un entrenamiento más rápido y efectivo. Nos quedamos con la activación que mejor val_accuracy obtenga para los siguientes experimentos.

### Experimento 3: Zero vs Glorot uniform


In [ ]:
# ---- MODELO CON INICIALIZACIÓN A CEROS ----
# Inicializar todos los pesos a cero es una mala práctica, ya que todas las
# neuronas aprenderían lo mismo (problema de simetría).
model_zeros = models.Sequential()
model_zeros.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3),
                               padding='same', kernel_initializer='zeros', name='Convolutiva-1'))
model_zeros.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_zeros.add(layers.Flatten())
model_zeros.add(layers.Dense(128, activation='relu', kernel_initializer='zeros', name='Hidden-Layer-1'))
model_zeros.add(layers.Dense(64, activation='relu', kernel_initializer='zeros', name='Hidden-Layer-2'))
model_zeros.add(layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer='zeros', name='Output-Layer'))

model_zeros.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_zeros = model_zeros.fit(x_train, y_train, epochs=20, batch_size=32,
                                 validation_split=0.2, callbacks=[early_stop])


In [ ]:
# ---- MODELO CON INICIALIZACIÓN GLOROT UNIFORM (por defecto en Keras) ----
# Glorot (Xavier) inicializa los pesos con valores aleatorios escalados según
# el número de entradas y salidas de cada capa, rompiendo la simetría.
model_glorot = models.Sequential()
model_glorot.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3),
                                padding='same', kernel_initializer='glorot_uniform', name='Convolutiva-1'))
model_glorot.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_glorot.add(layers.Flatten())
model_glorot.add(layers.Dense(128, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-Layer-1'))
model_glorot.add(layers.Dense(64, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-Layer-2'))
model_glorot.add(layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer='glorot_uniform', name='Output-Layer'))

model_glorot.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_glorot = model_glorot.fit(x_train, y_train, epochs=20, batch_size=32,
                                   validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparación visual de los resultados entre inicialización Zeros y Glorot Uniform
plot_compare_losses(history_zeros, history_glorot, name1="Zeros", name2="Glorot",
                    title="Comparación de Loss: Zeros vs Glorot Uniform")
plot_compare_accs(history_zeros, history_glorot, name1="Zeros", name2="Glorot",
                  title="Comparación de Accuracy: Zeros vs Glorot Uniform")

print(f"Zeros  - Val Accuracy: {max(history_zeros.history['val_accuracy']):.4f}, Val Loss: {min(history_zeros.history['val_loss']):.4f}")
print(f"Glorot - Val Accuracy: {max(history_glorot.history['val_accuracy']):.4f}, Val Loss: {min(history_glorot.history['val_loss']):.4f}")


**Conclusión Experimento 3:** La inicialización a ceros impide que la red aprenda correctamente debido al problema de simetría: todas las neuronas computan lo mismo, por lo que el modelo no puede aprender características diferenciadas. Glorot Uniform rompe esa simetría e inicializa los pesos de forma adecuada. Se espera un rendimiento muy superior con Glorot.

### Experimento 4 - Aleatoria Normal vs Glorot uniform


In [ ]:
# ---- MODELO CON INICIALIZACIÓN ALEATORIA NORMAL ----
# RandomNormal inicializa los pesos con una distribución normal estándar.
# A diferencia de Glorot, no escala según el tamaño de las capas.
model_random = models.Sequential()
model_random.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3),
                                padding='same', kernel_initializer='random_normal', name='Convolutiva-1'))
model_random.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_random.add(layers.Flatten())
model_random.add(layers.Dense(128, activation='relu', kernel_initializer='random_normal', name='Hidden-Layer-1'))
model_random.add(layers.Dense(64, activation='relu', kernel_initializer='random_normal', name='Hidden-Layer-2'))
model_random.add(layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer='random_normal', name='Output-Layer'))

model_random.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_random = model_random.fit(x_train, y_train, epochs=20, batch_size=32,
                                   validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Reutilizamos el modelo Glorot del experimento anterior para comparar.
# Si el kernel se ha reiniciado, volvemos a entrenar:
# (Si history_glorot ya está disponible del Exp.3, podemos usarlo directamente)

# Comparación visual
plot_compare_losses(history_random, history_glorot, name1="Random Normal", name2="Glorot",
                    title="Comparación de Loss: Random Normal vs Glorot Uniform")
plot_compare_accs(history_random, history_glorot, name1="Random Normal", name2="Glorot",
                  title="Comparación de Accuracy: Random Normal vs Glorot Uniform")

print(f"Random Normal - Val Accuracy: {max(history_random.history['val_accuracy']):.4f}, Val Loss: {min(history_random.history['val_loss']):.4f}")
print(f"Glorot Uniform - Val Accuracy: {max(history_glorot.history['val_accuracy']):.4f}, Val Loss: {min(history_glorot.history['val_loss']):.4f}")


**Conclusión Experimento 4:** Glorot Uniform suele ofrecer mejor rendimiento que Random Normal porque está diseñada específicamente para mantener la varianza de las activaciones y los gradientes estable a lo largo de las capas. Random Normal puede causar activaciones muy grandes o muy pequeñas dependiendo de la arquitectura de la red.

### Experimento 5 - SGD vs RMSprop


In [ ]:
# ---- MODELO CON OPTIMIZADOR SGD ----
# SGD (Stochastic Gradient Descent) es el optimizador más básico,
# actualiza los pesos proporcionalmente al gradiente.
model_sgd = models.Sequential()
model_sgd.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_sgd.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_sgd.add(layers.Flatten())
model_sgd.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_sgd.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_sgd.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_sgd.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_sgd = model_sgd.fit(x_train, y_train, epochs=20, batch_size=32,
                             validation_split=0.2, callbacks=[early_stop])


In [ ]:
# ---- MODELO CON OPTIMIZADOR RMSprop ----
# RMSprop adapta la tasa de aprendizaje para cada parámetro basándose en
# una media móvil de los gradientes al cuadrado. Suele converger más rápido.
model_rmsprop = models.Sequential()
model_rmsprop.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_rmsprop.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_rmsprop.add(layers.Flatten())
model_rmsprop.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_rmsprop.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_rmsprop.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_rmsprop.compile(optimizer=RMSprop(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_rmsprop = model_rmsprop.fit(x_train, y_train, epochs=20, batch_size=32,
                                     validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparación visual SGD vs RMSprop
plot_compare_losses(history_sgd, history_rmsprop, name1="SGD", name2="RMSprop",
                    title="Comparación de Loss: SGD vs RMSprop")
plot_compare_accs(history_sgd, history_rmsprop, name1="SGD", name2="RMSprop",
                  title="Comparación de Accuracy: SGD vs RMSprop")

print(f"SGD     - Val Accuracy: {max(history_sgd.history['val_accuracy']):.4f}, Val Loss: {min(history_sgd.history['val_loss']):.4f}")
print(f"RMSprop - Val Accuracy: {max(history_rmsprop.history['val_accuracy']):.4f}, Val Loss: {min(history_rmsprop.history['val_loss']):.4f}")


**Conclusión Experimento 5:** RMSprop suele converger más rápido que SGD ya que adapta la tasa de aprendizaje para cada parámetro. SGD usa una tasa de aprendizaje fija. Se espera que RMSprop alcance mayor accuracy en menos epochs.

### Experimento 6: SGD vs Adamax

Probar con learning_rate=0.002, beta_1=0.9, beta_2=0.999

In [ ]:
# ---- MODELO CON OPTIMIZADOR SGD (reutilizamos history_sgd del Exp. 5) ----

# ---- MODELO CON OPTIMIZADOR ADAMAX ----
# Adamax es una variante de Adam basada en la norma infinito.
# Configuramos con los hiperparámetros especificados.
model_adamax = models.Sequential()
model_adamax.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_adamax.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_adamax.add(layers.Flatten())
model_adamax.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_adamax.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_adamax.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

# Compilamos con Adamax usando los hiperparámetros indicados
model_adamax.compile(optimizer=Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_adamax = model_adamax.fit(x_train, y_train, epochs=20, batch_size=32,
                                   validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparación visual SGD vs Adamax
plot_compare_losses(history_sgd, history_adamax, name1="SGD", name2="Adamax",
                    title="Comparación de Loss: SGD vs Adamax")
plot_compare_accs(history_sgd, history_adamax, name1="SGD", name2="Adamax",
                  title="Comparación de Accuracy: SGD vs Adamax")

print(f"SGD    - Val Accuracy: {max(history_sgd.history['val_accuracy']):.4f}, Val Loss: {min(history_sgd.history['val_loss']):.4f}")
print(f"Adamax - Val Accuracy: {max(history_adamax.history['val_accuracy']):.4f}, Val Loss: {min(history_adamax.history['val_loss']):.4f}")


**Conclusión Experimento 6:** Adamax es una variante de Adam más robusta ante gradientes grandes. Al adaptar la tasa de aprendizaje, converge generalmente más rápido que SGD y suele obtener mejor accuracy. Se elige el mejor optimizador para los siguientes experimentos.

### Experimento 7: Aumento batch size 512

In [ ]:
# ---- MODELO CON BATCH SIZE 512 ----
# Aumentamos el batch_size de 32 a 512. Un batch más grande hace que
# las actualizaciones sean más estables pero menos frecuentes por epoch.
# También puede causar peor generalización.

# Usamos el mejor optimizador encontrado hasta ahora.
# (Usaremos SGD por consistencia con la estructura del laboratorio)
model_bs512 = models.Sequential()
model_bs512.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_bs512.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_bs512.add(layers.Flatten())
model_bs512.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_bs512.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_bs512.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_bs512.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

# Entrenamos con batch_size=512 en lugar de 32
history_bs512 = model_bs512.fit(x_train, y_train, epochs=20, batch_size=512,
                                 validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparamos batch_size=32 (history_sgd) contra batch_size=512
plot_compare_losses(history_sgd, history_bs512, name1="Batch 32", name2="Batch 512",
                    title="Comparación de Loss: Batch 32 vs Batch 512")
plot_compare_accs(history_sgd, history_bs512, name1="Batch 32", name2="Batch 512",
                  title="Comparación de Accuracy: Batch 32 vs Batch 512")

print(f"Batch 32  - Val Accuracy: {max(history_sgd.history['val_accuracy']):.4f}, Val Loss: {min(history_sgd.history['val_loss']):.4f}")
print(f"Batch 512 - Val Accuracy: {max(history_bs512.history['val_accuracy']):.4f}, Val Loss: {min(history_bs512.history['val_loss']):.4f}")


**Conclusión Experimento 7:** Un batch size mayor (512) suele producir actualizaciones más estables pero menos frecuentes, lo que puede llevar a una convergencia más lenta. Además, batch sizes muy grandes tienden a generalizar peor que batch sizes más pequeños. Se espera que batch_size=32 ofrezca mejor rendimiento.

### Experimento 8 - Aplicar BatchNormalization

In [ ]:
# ---- MODELO CON BATCH NORMALIZATION ----
# BatchNormalization normaliza las activaciones de cada capa, lo que:
# - Estabiliza el entrenamiento
# - Permite tasas de aprendizaje más altas
# - Actúa como regularizador ligero
model_bn = models.Sequential()
model_bn.add(layers.Conv2D(64, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_bn.add(layers.BatchNormalization())  # BatchNorm después de la convolución
model_bn.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_bn.add(layers.Flatten())
model_bn.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model_bn.add(layers.BatchNormalization())  # BatchNorm después de la primera capa densa
model_bn.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model_bn.add(layers.BatchNormalization())  # BatchNorm después de la segunda capa densa
model_bn.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_bn.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_bn = model_bn.fit(x_train, y_train, epochs=20, batch_size=32,
                           validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparamos modelo base (sin BatchNorm) vs modelo con BatchNorm
plot_compare_losses(history_sgd, history_bn, name1="Sin BatchNorm", name2="Con BatchNorm",
                    title="Comparación de Loss: Sin vs Con BatchNormalization")
plot_compare_accs(history_sgd, history_bn, name1="Sin BatchNorm", name2="Con BatchNorm",
                  title="Comparación de Accuracy: Sin vs Con BatchNormalization")

print(f"Sin BatchNorm - Val Accuracy: {max(history_sgd.history['val_accuracy']):.4f}, Val Loss: {min(history_sgd.history['val_loss']):.4f}")
print(f"Con BatchNorm - Val Accuracy: {max(history_bn.history['val_accuracy']):.4f}, Val Loss: {min(history_bn.history['val_loss']):.4f}")


**Conclusión Experimento 8:** BatchNormalization suele mejorar significativamente el entrenamiento al reducir el desplazamiento interno de covariables (internal covariate shift). Se espera una convergencia más rápida y posiblemente mejor accuracy en validación.

### Experimento 9 - Aumentar el número de parámetros por capa

Aumentar de la siguiente manera:

*   512 a la Conv2D
*   512 a la primera Dense
*   256 a la segunda Dense


In [ ]:
# ---- MODELO CON MÁS PARÁMETROS POR CAPA ----
# Aumentamos la capacidad del modelo incrementando el número de filtros/neuronas:
# Conv2D: 64 -> 512 filtros
# Dense 1: 128 -> 512 neuronas
# Dense 2: 64 -> 256 neuronas
model_big = models.Sequential()
model_big.add(layers.Conv2D(512, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_big.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_big.add(layers.Flatten())
model_big.add(layers.Dense(512, activation='relu', name='Hidden-Layer-1'))
model_big.add(layers.Dense(256, activation='relu', name='Hidden-Layer-2'))
model_big.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_big.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_big.summary()  # Muestra el resumen para ver el aumento de parámetros

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_big = model_big.fit(x_train, y_train, epochs=20, batch_size=32,
                             validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparamos modelo base (pequeño) vs modelo con más parámetros
plot_compare_losses(history_sgd, history_big, name1="Base (64-128-64)", name2="Grande (512-512-256)",
                    title="Comparación de Loss: Modelo Base vs Modelo Grande")
plot_compare_accs(history_sgd, history_big, name1="Base (64-128-64)", name2="Grande (512-512-256)",
                  title="Comparación de Accuracy: Modelo Base vs Modelo Grande")

print(f"Base   - Val Accuracy: {max(history_sgd.history['val_accuracy']):.4f}, Val Loss: {min(history_sgd.history['val_loss']):.4f}")
print(f"Grande - Val Accuracy: {max(history_big.history['val_accuracy']):.4f}, Val Loss: {min(history_big.history['val_loss']):.4f}")


**Conclusión Experimento 9:** Un modelo con más parámetros tiene mayor capacidad de aprendizaje, pero también es más propenso al overfitting. Es posible que el modelo grande obtenga mayor accuracy en training pero peor en validación si no se aplican técnicas de regularización.

### Experimento 10 - Aplicar Dropout 0.2

In [ ]:
# ---- MODELO GRANDE CON DROPOUT 0.2 ----
# Aplicamos Dropout del 20% al modelo grande del Exp. 9 para regularizar.
# Dropout desactiva aleatoriamente un porcentaje de neuronas durante el
# entrenamiento, lo que reduce el overfitting al forzar a la red
# a no depender excesivamente de neuronas individuales.
model_dropout = models.Sequential()
model_dropout.add(layers.Conv2D(512, (2, 2), activation='relu', input_shape=(32, 32, 3), padding='same', name='Convolutiva-1'))
model_dropout.add(layers.MaxPooling2D(pool_size=(2, 2), name='MaxPooling-1'))
model_dropout.add(layers.Dropout(0.2))  # Dropout del 20% después del MaxPooling
model_dropout.add(layers.Flatten())
model_dropout.add(layers.Dense(512, activation='relu', name='Hidden-Layer-1'))
model_dropout.add(layers.Dropout(0.2))  # Dropout del 20% después de la primera Dense
model_dropout.add(layers.Dense(256, activation='relu', name='Hidden-Layer-2'))
model_dropout.add(layers.Dropout(0.2))  # Dropout del 20% después de la segunda Dense
model_dropout.add(layers.Dense(NUM_CLASSES, activation='softmax', name='Output-Layer'))

model_dropout.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_dropout = model_dropout.fit(x_train, y_train, epochs=20, batch_size=32,
                                     validation_split=0.2, callbacks=[early_stop])


In [ ]:
# Comparamos modelo grande sin Dropout vs con Dropout
plot_compare_losses(history_big, history_dropout, name1="Sin Dropout", name2="Con Dropout 0.2",
                    title="Comparación de Loss: Sin vs Con Dropout 0.2")
plot_compare_accs(history_big, history_dropout, name1="Sin Dropout", name2="Con Dropout 0.2",
                  title="Comparación de Accuracy: Sin vs Con Dropout 0.2")

print(f"Sin Dropout - Val Accuracy: {max(history_big.history['val_accuracy']):.4f}, Val Loss: {min(history_big.history['val_loss']):.4f}")
print(f"Con Dropout - Val Accuracy: {max(history_dropout.history['val_accuracy']):.4f}, Val Loss: {min(history_dropout.history['val_loss']):.4f}")


**Conclusión Experimento 10:** Dropout actúa como técnica de regularización desactivando neuronas aleatoriamente. Se espera que reduzca la diferencia entre accuracy de entrenamiento y validación, mitigando el overfitting del modelo grande.

### Anexos

Si os encontráis alguna anomalía mientras realizáis el laboratorio, describirla en este punto, motivos del problema y solución.


In [ ]:
# ---- TABLA RESUMEN DE TODOS LOS EXPERIMENTOS ----
# Recopilamos los mejores resultados de validación de cada experimento
print("=" * 70)
print(f"{'Experimento':<40} {'Val Acc':>10} {'Val Loss':>10}")
print("=" * 70)

results = [
    ('Exp 2 - ReLU', history_relu),
    ('Exp 2 - Tanh', history_tanh),
    ('Exp 3 - Zeros', history_zeros),
    ('Exp 3 - Glorot Uniform', history_glorot),
    ('Exp 4 - Random Normal', history_random),
    ('Exp 5 - SGD', history_sgd),
    ('Exp 5 - RMSprop', history_rmsprop),
    ('Exp 6 - Adamax', history_adamax),
    ('Exp 7 - Batch 512', history_bs512),
    ('Exp 8 - BatchNorm', history_bn),
    ('Exp 9 - Modelo Grande', history_big),
    ('Exp 10 - Dropout 0.2', history_dropout),
]

for name, hist in results:
    best_acc = max(hist.history['val_accuracy'])
    best_loss = min(hist.history['val_loss'])
    print(f"{name:<40} {best_acc:>10.4f} {best_loss:>10.4f}")

print("=" * 70)


In [ ]:
# ---- EVALUACIÓN FINAL EN EL TEST SET ----
# Evaluamos el modelo que consideremos mejor en los datos de test
# para obtener la métrica final del laboratorio.

# Nota: este score del test set NO debe utilizarse para seleccionar
# hiperparámetros. Solo se evalúa al final con el modelo ya elegido.

print("\n=== Evaluación en el Test Set ===")
print("\nModelo ReLU (Exp 2):")
test_loss, test_acc = model_relu.evaluate(x_test, y_test, verbose=0)
print(f"  Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

print("\nModelo con Dropout (Exp 10):")
test_loss, test_acc = model_dropout.evaluate(x_test, y_test, verbose=0)
print(f"  Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")
